# Real Car Training Notebook

在 Jupyter 中运行真实小车的测试和训练，支持实时画面显示。

## 功能
1. CTE 估算测试与可视化
2. 实时摄像头画面显示
3. 训练过程监控


## 1. 导入依赖


In [ ]:
import sys
import time
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

# 添加父目录到路径
sys.path.insert(0, str(Path.cwd().parent))

from real_car_env import RealJetRacerEnv
from cte_estimator import VisualCTEEstimator

%matplotlib inline
plt.rcParams['figure.figsize'] = [14, 5]

print("✅ 依赖导入完成！")


## 2. Notebook 可视化工具类


In [ ]:
class NotebookVisualizer:
    """Notebook 实时可视化工具"""
    
    def __init__(self):
        self.cte_history = []
        self.reward_history = []
        self.confidence_history = []
        
    def reset_history(self):
        """重置历史记录"""
        self.cte_history = []
        self.reward_history = []
        self.confidence_history = []
    
    def show_frame(self, frame_bgr, cte=None, confidence=None, reward=None, 
                   debug_image=None, mask_image=None, title="Camera"):
        """显示单帧图像"""
        clear_output(wait=True)
        
        # 确保frame_bgr是有效的
        if frame_bgr is None:
            frame_bgr = np.zeros((240, 320, 3), dtype=np.uint8)
        frame_bgr = np.asarray(frame_bgr)
        if len(frame_bgr.shape) == 2:
            frame_bgr = cv2.cvtColor(frame_bgr, cv2.COLOR_GRAY2BGR)
        
        n_plots = 1  # 原始图像（总是显示）
        if debug_image is not None:
            n_plots += 1
        if mask_image is not None:
            n_plots += 1
        if len(self.cte_history) > 0:
            n_plots += 1
        
        fig, axes = plt.subplots(1, n_plots, figsize=(5 * n_plots, 4))
        if n_plots == 1:
            axes = [axes]
        
        plot_idx = 0
        
        # 原始图像（总是第一个显示）
        try:
            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            axes[plot_idx].imshow(frame_rgb)
        except Exception as e:
            # 如果转换失败，尝试直接显示
            axes[plot_idx].imshow(frame_bgr)
        
        info_text = title
        if cte is not None:
            info_text += f"\nCTE: {cte:+.2f}"
        if confidence is not None:
            info_text += f" | Conf: {confidence:.2f}"
        if reward is not None:
            info_text += f" | R: {reward:.2f}"
        axes[plot_idx].set_title(info_text)
        axes[plot_idx].axis('off')
        plot_idx += 1
        
        # Debug 图像 (CTE检测结果)
        if debug_image is not None:
            axes[plot_idx].imshow(cv2.cvtColor(debug_image, cv2.COLOR_BGR2RGB))
            axes[plot_idx].set_title("CTE Detection")
            axes[plot_idx].axis('off')
            plot_idx += 1
        
        # Mask 图像
        if mask_image is not None:
            axes[plot_idx].imshow(mask_image, cmap='gray')
            axes[plot_idx].set_title("Mask View")
            axes[plot_idx].axis('off')
            plot_idx += 1
        
        # 历史曲线
        if len(self.cte_history) > 0:
            ax_hist = axes[-1]
            ax_hist.plot(self.cte_history, 'b-', label='CTE', linewidth=1.5)
            ax_hist.axhline(y=0, color='g', linestyle='--', alpha=0.5)
            ax_hist.axhline(y=3, color='r', linestyle='--', alpha=0.5)
            ax_hist.axhline(y=-3, color='r', linestyle='--', alpha=0.5)
            ax_hist.set_ylim(-4, 4)
            ax_hist.set_xlabel('Step')
            ax_hist.set_ylabel('CTE')
            ax_hist.set_title(f'CTE History (n={len(self.cte_history)})')
            ax_hist.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
    
    def update_history(self, cte, reward=None, confidence=None):
        """更新历史记录"""
        self.cte_history.append(cte)
        if reward is not None:
            self.reward_history.append(reward)
        if confidence is not None:
            self.confidence_history.append(confidence)
    
    def show_summary(self):
        """显示统计摘要"""
        if not self.cte_history:
            print("没有数据")
            return
        
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        
        axes[0].plot(self.cte_history, 'b-', linewidth=1)
        axes[0].axhline(y=0, color='g', linestyle='--', alpha=0.5)
        axes[0].fill_between(range(len(self.cte_history)), self.cte_history, alpha=0.3)
        axes[0].set_title(f'CTE History\nMean: {np.mean(self.cte_history):.3f}, Std: {np.std(self.cte_history):.3f}')
        axes[0].set_xlabel('Step')
        axes[0].set_ylabel('CTE')
        axes[0].grid(True, alpha=0.3)
        
        axes[1].hist(self.cte_history, bins=30, edgecolor='black', alpha=0.7)
        axes[1].axvline(x=0, color='g', linestyle='--', linewidth=2)
        axes[1].set_title('CTE Distribution')
        axes[1].set_xlabel('CTE')
        axes[1].set_ylabel('Count')
        
        plt.tight_layout()
        plt.show()
        
        print(f"\n统计:")
        print(f"  步数: {len(self.cte_history)}")
        print(f"  平均 CTE: {np.mean(self.cte_history):.3f}")
        print(f"  CTE 标准差: {np.std(self.cte_history):.3f}")
        if self.reward_history:
            print(f"  总奖励: {np.sum(self.reward_history):.1f}")

viz = NotebookVisualizer()
print("✅ 可视化工具创建完成")


## 3. 配置与初始化环境


In [ ]:
# ============ 配置 ============
CONFIG = {
    "cam_type": "csi",  # "csi" 或 "usb"
    "cam_width": 320,
    "cam_height": 240,
    "obs_width": 84,
    "obs_height": 84,
    "obs_mode": "perspective",
    "cte_estimator": "centerline_tracking",  # "edge_detection", "centerline_tracking"
    "max_cte": 3.0,
    "throttle_gain": 0.5,
    "steering_gain": 0.5,
    "max_episode_steps": 500,
}

# 创建环境
env = RealJetRacerEnv(
    cam_type=CONFIG["cam_type"],
    cam_width=CONFIG["cam_width"],
    cam_height=CONFIG["cam_height"],
    obs_width=CONFIG["obs_width"],
    obs_height=CONFIG["obs_height"],
    obs_mode=CONFIG["obs_mode"],
    cte_estimator=CONFIG["cte_estimator"],
    max_cte=CONFIG["max_cte"],
    throttle_gain=CONFIG["throttle_gain"],
    steering_gain=CONFIG["steering_gain"],
    max_episode_steps=CONFIG["max_episode_steps"],
    render_mode=None,
)
print("✅ 环境创建完成")


## 4. CTE 估算测试 (使用测试图片或摄像头)


In [ ]:
def test_cte(n_frames=10, delay=0.5, use_camera=False):
    """测试 CTE 估算"""
    print(f"开始 CTE 测试 (use_camera={use_camera})...")
    
    # 只在需要相机时才初始化硬件
    if use_camera:
        env._init_hardware()
    
    viz.reset_history()
    
    # 获取测试图片
    test_images = sorted(Path('../real_road_data').glob('*.jpg'))
    
    for i in range(n_frames):
        if use_camera and env._camera is not None:
            frame = env._camera.read()
        elif test_images:
            frame = cv2.imread(str(test_images[i % len(test_images)]))
        else:
            frame = np.zeros((CONFIG['cam_height'], CONFIG['cam_width'], 3), dtype=np.uint8)
        
        # 先调用estimate，这样last_mask_image会被设置
        cte, confidence = env.cte_estimator.estimate(frame)
        viz.update_history(cte, confidence=confidence)
        
        # 获取CTE估计使用的mask（不是obs_mode的mask）
        # edge_detection方法：显示Canny边缘检测结果
        # centerline_tracking方法：显示HSV mask（中心线检测）
        # 注意：last_mask_image在estimate()之后才会被设置
        cte_mask_image = getattr(env.cte_estimator, 'last_mask_image', None)
        
        viz.show_frame(
            frame, cte=cte, confidence=confidence,
            debug_image=env.cte_estimator.last_debug_image,
            mask_image=cte_mask_image,
            title=f"Frame {i+1}/{n_frames}"
        )
        time.sleep(delay)
    
    viz.show_summary()

# 运行测试 (使用 real_road_data 中的图片)
test_cte(n_frames=4, delay=1.0, use_camera=False)


## 5. Episode 测试 (实际运行小车)


In [ ]:
def run_episode(action_mode='zero', max_steps=50, fps=10):
    """
    运行一个 episode
    action_mode: 'zero'=不动, 'forward'=直行, 'random'=随机
    """
    print(f"运行 Episode (action_mode={action_mode})...")
    viz.reset_history()
    
    # 注意: reset() 会等待用户按 Enter
    obs = env.reset()
    
    total_reward = 0
    done = False
    step = 0
    
    while not done and step < max_steps:
        if action_mode == 'zero':
            action = np.array([0.0, 0.0], dtype=np.float32)
        elif action_mode == 'forward':
            action = np.array([0.2, 0.0], dtype=np.float32)
        else:
            action = env.action_space.sample()
            action[0] = np.clip(action[0], 0.0, 0.3)
        
        obs, reward, done, info = env.step(action)
        total_reward += reward
        viz.update_history(info['cte'], reward=reward)
        
        if step % 5 == 0:
            if env._camera is not None:
                frame = env._camera.read()
            else:
                frame = np.zeros((CONFIG['cam_height'], CONFIG['cam_width'], 3), dtype=np.uint8)
            
            # 获取CTE估计使用的mask（不是obs_mode的mask）
            cte_mask_image = env.cte_estimator.last_mask_image
            
            viz.show_frame(frame, cte=info['cte'], reward=reward,
                          debug_image=env.cte_estimator.last_debug_image,
                          mask_image=cte_mask_image,
                          title=f"Step {step} | T={action[0]:.2f} S={action[1]:.2f}")
        
        step += 1
        time.sleep(1.0 / fps)
    
    print(f"\nEpisode 结束! 步数: {step}, 总奖励: {total_reward:.2f}")
    viz.show_summary()

# 运行静态测试 (取消注释运行)
# run_episode(action_mode='zero', max_steps=20)


## 6. 使用训练好的模型测试


In [ ]:
def run_with_model(model_path, max_steps=100, fps=10):
    """使用训练好的模型运行"""
    from stable_baselines3 import PPO
    
    print(f"加载模型: {model_path}")
    model = PPO.load(model_path)
    
    viz.reset_history()
    obs = env.reset()
    
    total_reward = 0
    done = False
    step = 0
    
    while not done and step < max_steps:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, done, info = env.step(action)
        total_reward += reward
        viz.update_history(info['cte'], reward=reward)
        
        if step % 5 == 0:
            if env._camera is not None:
                frame = env._camera.read()
            else:
                frame = np.zeros((CONFIG['cam_height'], CONFIG['cam_width'], 3), dtype=np.uint8)
            
            # 获取CTE估计使用的mask（不是obs_mode的mask）
            cte_mask_image = env.cte_estimator.last_mask_image
            
            viz.show_frame(frame, cte=info['cte'], reward=reward,
                          debug_image=env.cte_estimator.last_debug_image,
                          mask_image=cte_mask_image,
                          title=f"Model Run | Step {step}")
        
        step += 1
        time.sleep(1.0 / fps)
    
    print(f"运行结束! 总奖励: {total_reward:.2f}")
    viz.show_summary()

# 使用模型测试 (取消注释运行)
# run_with_model('../logs/shimmy_ppo_85000_steps.zip')


## 7. 清理资源


In [ ]:
def cleanup():
    """清理资源"""
    global env
    if env is not None:
        env.close()
        print("✅ 环境已关闭")

# 取消注释执行清理
# cleanup()
